# NBA 3.1 - 08 Census / SDoH features

**Assessment:** `NBA 3.0 Static Feature Analysis/08_2_census_data_assessment.md` | **Universe:** `HCP_Universe_without_call_hcps` (294,727) | **Cohorts:** 28 claims cohorts c1-c28 | **Writers (signal only):** KER-10 `active_nbrx_writers` in `kerendia_metric_views.mv_kerendia_nbrx` | **Output:** `kerendia_nba_3_1_volume/.../notebook_output/hcp_features/census_features`

_Summary is filled in after the run (see the gate cell for the live results)._

## Decisions taken in this notebook

The feature list follows `08_2_census_data_assessment.md` section 4 (Build and Test rows) and section 6 (recommended order). The layout follows `33_colocation_idn_features` (same universe, cohorts, keys and output folder). Decisions below were taken while building, where 08_2 left a choice open or the 3.1 pipeline needed one.

| # | Topic | Decision | Why |
| --- | --- | --- | --- |
| N1 | Source | Read `whc_data_zone1_prod.dem_zip_sdoh_acs20` from UC instead of `SDoH Walgreens v2.csv` | 08_2 source 2 / section 6 step 1. The CSV is checked once below to be identical (0 differing cells) |
| N2 | Join key | HCP ZIP joined on **ZIPCODE**, not ZCTA | 08_2 data issue 1: the ZCTA join in the 3.1 draft loses 4,443 HCPs |
| N3 | HCP ZIP | `HCP_ZIP` of the BEST_AFFL `BRAND_RANK = 1` row (UC `pace_ker_in_house_schema` copy), first 5 characters, left-padded with 0 | 08_2 method step 2; same bridge as colocation F1 |
| N4 | WHITE | `ACS_PCT_WHITE_NONHISP_ZC` **replaces** `ACS_PCT_WHITE_ZC`; `ACS_PCT_POSTHS_ED_ZC` dropped. 21 ZIP columns are built | 08_2 section 4 rows 1 and 6 (WHITE_NONHISP rho -0.085 vs -0.075; POSTHS_ED = sum of three columns) |
| N5 | One county measure | `COUNTY_DEP_POPW` is Build; `COUNTY_SDOH_LT_HS` is Test | 08_2 section 4: "Build one county measure; test the other". The population-weighted version also uses the new population column |
| N6 | Roll-ups | Territory and county means are taken over **one row per ZCTA** in each area, only CR1 rows of `zip_terr_Q2_2026.csv`. Territory and WHITE roll-ups are unweighted, `COUNTY_DEP_POPW` is weighted by `ACS_TOT_POP_WT_ZC` | 08_2 section 3.3 and data issues 3, 4, 6. Q2 2026 is the file the assessment used; it is frozen here so training and inference use one mapping |
| N7 | Deprivation index | Mean z-score of LT_HS, UNEMPLOY, UNINSURED_BELOW64, TRANSPORT and minus PER_CAPITA_INC; mean and sd over one row per ZCTA | Same definition as the 3.1 draft and 08_2 section 3.2 |
| N8 | Missing states | No ZIP or no ACS value: all ZIP features null, `SDOH_MISSING_FLAG = 1`. `NO_CBSA_FLAG` is null only when there is no ZIP | 08_2 section 2 and data issue 7 (no-ZIP HCPs write at 4.8%, so missing is informative) |
| N9 | Grain and keys | One row per `PROVIDER_ID` x `cohort_id` (28 claims cohorts); `SVC_DT_M` = cohort `lookback_end_date`; `HCP_COHORT_ID = PROVIDER_ID_cohort_id`. Values are the same in every cohort | Same keys as `33_colocation_idn_features` (its N3). Census data is static and the HCP ZIP is a current snapshot (D7 of the colocation plan applies) |
| N10 | Output path | `kerendia_nba_3_1_volume/hcp_level_model/dev_pipeline/notebook_output/hcp_features/census_features` | Same folder as colocation; never write to the 3.0 volume or over the 3.1 draft output |
| N11 | Signal restatement | KER-10 writers (D8): cohort c17, look-back 2025-07 to 2025-09, outcome = KER-10 writer in 2025-10 to 2026-09; new-writer view = not counted by KER-10 in the look-back | Same window as colocation N13. LRx V7 is not used |
| N12 | Population | `ACS_TOT_POP_WT_ZC` is used as the county weight; `LOG_POP` (= log1p(population)) is the Test feature | 08_2 section 4: "Build as weight, Test as feature" |

In [0]:
import json
import numpy as np
import pandas as pd
import pyspark.sql.functions as F
from functools import reduce

CAT = "`ph-com-ai-us-prod-catalog-2553213839599380`"
VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume"
VOL30 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume"

HCP_UNIVERSE_PATH = f"{VOL31}/HCP_Universe_without_call_hcps"        # 294,727 HCPs
COHORTS_DIR = f"{VOL31}/cohorts/claims"                              # 28 claims cohorts (N9)
MV_NBRX = f"{CAT}.`kerendia_metric_views`.`mv_kerendia_nbrx`"        # KER-10 writers, signal check only (N11)
MAPPING = f"{CAT}.kerendia_nba_3.source_provider_bhid_npi_mapping"
BEST_AFFL = f"{CAT}.pace_ker_in_house_schema.cust_mdm_best_affl_publish_pd_outp_static"
ACS20 = f"{CAT}.whc_data_zone1_prod.dem_zip_sdoh_acs20"              # N1
CBSA_GEO = f"{CAT}.whc_data_zone1_prod.vms_laad_dim_geo"
INPUT_DIR = f"{VOL30}/hcp_level_model/ops_pipeline/input_files"
ZIP_TERR_PATH = f"{INPUT_DIR}/zip_terr_Q2_2026.csv"                  # N6 (frozen)
SDOH_CSV_PATH = f"{INPUT_DIR}/SDoH Walgreens v2.csv"                 # NBA 3.0 file, equality check only

OUT_DIR = f"{VOL31}/hcp_level_model/dev_pipeline/notebook_output/hcp_features"
OUTPUT_PATH = f"{OUT_DIR}/census_features"                           # N10
assert "kerendia_nba_3_1_volume" in OUTPUT_PATH, "never write outside the 3.1 volume"

# The 22 NBA 3.0 ACS columns
BASE22 = ["ACS_PCT_ASIAN_ZC", "ACS_PCT_BLACK_ZC", "ACS_PCT_HISPANIC_ZC", "ACS_PCT_OTHER_RACE_ZC", "ACS_PCT_WHITE_ZC",
          "ACS_PCT_EDUC_ZC", "ACS_PCT_TRANSPORT_ZC", "ACS_PCT_UNEMPLOY_ZC", "ACS_PER_CAPITA_INC_ZC",
          "ACS_PCT_POV_ASIAN_ZC", "ACS_PCT_POV_BLACK_ZC", "ACS_PCT_POV_HISPANIC_ZC", "ACS_PCT_POV_WHITE_ZC",
          "ACS_PCT_COLLEGE_ASSOCIATE_DGR_ZC", "ACS_PCT_BACHELOR_DGR_ZC", "ACS_PCT_GRADUATE_DGR_ZC", "ACS_PCT_HS_GRADUATE_ZC",
          "ACS_PCT_LT_HS_ZC", "ACS_PCT_POSTHS_ED_ZC", "ACS_PCT_PUBL_TRANSIT_ZC", "ACS_PCT_UNINSURED_ZC", "ACS_PCT_UNINSURED_BELOW64_ZC"]
# N4: drop POSTHS_ED, replace WHITE with WHITE_NONHISP -> 21 ZIP columns
ZIP_BUILD = [c for c in BASE22 if c not in ("ACS_PCT_POSTHS_ED_ZC", "ACS_PCT_WHITE_ZC")] + ["ACS_PCT_WHITE_NONHISP_ZC"]
ZIP_TEST = ["ACS_PCT_MEDICAID_ANY_ZC", "ACS_PCT_MARRIED_SP_AB_F_ZC", "ACS_PCT_1UP_PERS_1ROOM_ZC", "ACS_PCT_SELF_MDCR_ABOVE35_ZC"]
DEP_COLS = ["ACS_PCT_LT_HS_ZC", "ACS_PCT_UNEMPLOY_ZC", "ACS_PCT_UNINSURED_BELOW64_ZC", "ACS_PCT_TRANSPORT_ZC", "ACS_PER_CAPITA_INC_ZC"]
DEP_SIGN = {"ACS_PER_CAPITA_INC_ZC": -1}
POP_COL = "ACS_TOT_POP_WT_ZC"

BUILD_COLS = ZIP_BUILD + ["SDOH_MISSING_FLAG", "TERR_SDOH_LT_HS", "TERR_SDOH_WHITE", "COUNTY_DEP_POPW", "NO_CBSA_FLAG"]
TEST_COLS = ZIP_TEST + ["SDOH_DEPRIVATION_INDEX", "TERR_DEPRIVATION_MEAN", "TERR_SDOH_HISPANIC",
                        "COUNTY_SDOH_LT_HS", "COUNTY_SDOH_WHITE", "LOG_POP"]
FEATURE_COLS = BUILD_COLS + TEST_COLS
FLAG_COLS = ["SDOH_MISSING_FLAG", "NO_CBSA_FLAG"]
assert len(ZIP_BUILD) == 21 and len(BUILD_COLS) == 26 and len(TEST_COLS) == 10
KEYS = ["PROVIDER_ID", "cohort_id", "SVC_DT_M", "HCP_COHORT_ID"]

# Every check writes its result here; the gate cell and the summary read it
CHECKS = {}
def record(check_id, passed, detail):
    CHECKS[check_id] = {"passed": bool(passed), "detail": detail}
    print(f"[{'PASS' if passed else 'FAIL'}] {check_id}: {detail}")

print(f"{len(FEATURE_COLS)} feature columns ({len(BUILD_COLS)} Build + {len(TEST_COLS)} Test) -> {OUTPUT_PATH}")

In [0]:
cohorts = pd.read_parquet(f"{COHORTS_DIR}/cohorts.parquet")
cohorts = cohorts.sort_values("cohort_id", key=lambda s: s.str[1:].astype(int)).reset_index(drop=True)
with open(f"{COHORTS_DIR}/metadata.json") as f:
    meta = json.load(f)
assert meta["feature_family"] == "claims" and len(cohorts) == meta["cohort_count"] == 28
for c in ["prediction_month", "lookback_end_date"]:
    cohorts[c] = pd.to_datetime(cohorts[c]).dt.date
coh = spark.createDataFrame(cohorts[["cohort_id", "dataset", "prediction_month", "lookback_end_date"]])

universe = spark.read.parquet(HCP_UNIVERSE_PATH).select(F.col("PROVIDER_ID").cast("string")).distinct()
n_hcp = universe.count()
assert n_hcp == 294727, n_hcp
print(f"HCP universe {n_hcp:,} | {len(cohorts)} claims cohorts {cohorts.cohort_id.iloc[0]}..{cohorts.cohort_id.iloc[-1]}")
display(cohorts[["cohort_id", "dataset", "prediction_month", "lookback_end_date"]])

## Phase 0: Source and coverage checks

The source swap (N1), the join fix (N2) and the HCP ZIP spine (N3) are checked against the counts in `08_2` section 2 before any feature is built. Results go to `CHECKS`.

In [0]:
need = sorted(set(BASE22 + ZIP_BUILD + ZIP_TEST + DEP_COLS + [POP_COL]))
acs_raw = spark.table(ACS20)
missing_cols = [c for c in need if c not in acs_raw.columns]
assert not missing_cols, missing_cols
acs = acs_raw.select(F.lpad(F.col("ZIPCODE").cast("string"), 5, "0").alias("ZIPCODE"),
                     F.lpad(F.col("ZCTA").cast("string"), 5, "0").alias("ZCTA"),
                     *[F.col(c).cast("double").alias(c) for c in need])
n_acs, n_zip = acs.count(), acs.select("ZIPCODE").distinct().count()
n_zcta = acs.select("ZCTA").distinct().count()
record("S1 acs20 one row per ZIPCODE", n_acs == n_zip, f"{n_acs:,} rows, {n_zip:,} ZIPCODEs, {n_zcta:,} distinct ZCTAs (08_2: 41,011 / 32,441)")

# N1: the 21 kept NBA 3.0 columns (and WHITE) must be identical to the CSV used in NBA 3.0
csv = spark.read.csv(SDOH_CSV_PATH, header=True)
cmp_cols = [c for c in BASE22 if c != "ACS_PCT_POSTHS_ED_ZC"]
c2 = csv.select(F.lpad("ZIPCODE", 5, "0").alias("ZIPCODE"), *[F.col(c).cast("double").alias(f"{c}__csv") for c in cmp_cols])
j = acs.join(c2, "ZIPCODE", "inner")
n_join = j.count()
n_diff = j.select(sum(F.when(~F.col(c).eqNullSafe(F.col(f"{c}__csv")), 1).otherwise(0) for c in cmp_cols).alias("d")).agg(F.sum("d")).collect()[0][0]
record("N1 acs20 = Walgreens CSV", n_join == n_acs and n_diff == 0,
       f"{n_join:,} ZIPs joined, {len(cmp_cols)} columns compared, differing cells: {n_diff}")

In [0]:
mp = spark.table(MAPPING).select(F.col("PROVIDER_ID").cast("string"), F.col("BH_ID").cast("string")).dropna().distinct()
assert mp.groupBy("PROVIDER_ID").count().filter("count > 1").count() == 0, "PROVIDER_ID maps to several BH_IDs"

best_all = (spark.table(BEST_AFFL).filter("BRAND_RANK = 1")
            .select(F.col("BH_ID").cast("string"),
                    F.lpad(F.substring(F.trim(F.col("HCP_ZIP").cast("string")), 1, 5), 5, "0").alias("ZIPCODE")).distinct())
multi_zip = best_all.groupBy("BH_ID").count().filter("count > 1").count()
record("S2 one rank-1 ZIP per BH_ID", multi_zip == 0, f"BH_IDs with more than one rank-1 ZIP: {multi_zip}")

spine = universe.join(mp, "PROVIDER_ID", "left").join(best_all, "BH_ID", "left").select("PROVIDER_ID", "BH_ID", "ZIPCODE")
acs_zip = acs.select("ZIPCODE").withColumn("_in_file", F.lit(1))
acs_zcta = acs.select(F.col("ZCTA").alias("ZIPCODE")).distinct().withColumn("_in_zcta", F.lit(1))
cov = (spine.join(acs_zip, "ZIPCODE", "left").join(acs_zcta, "ZIPCODE", "left")
       .agg(F.count("*").alias("rows"), F.countDistinct("PROVIDER_ID").alias("hcps"), F.count("BH_ID").alias("with_bh"),
            F.count("ZIPCODE").alias("with_zip"), F.count("_in_file").alias("zip_in_file"), F.count("_in_zcta").alias("zip_as_zcta"))
       .collect()[0].asDict())
print(cov)
expected = {"rows": 294727, "hcps": 294727, "with_bh": 294578, "with_zip": 282881, "zip_in_file": 282857, "zip_as_zcta": 278414}
record("S2 coverage = 08_2 section 2", all(cov[k] == v for k, v in expected.items()),
       f"BH_ID {cov['with_bh']:,} (294,578), ZIP {cov['with_zip']:,} (282,881), ZIPCODE join {cov['zip_in_file']:,} (282,857), "
       f"ZCTA join would give {cov['zip_as_zcta']:,} (278,414): {cov['zip_in_file'] - cov['zip_as_zcta']:,} HCPs recovered by N2")

## Phase 1: Build features

ZIP features (21 Build + 4 Test), the deprivation index (N7), territory and county roll-ups over one row per ZCTA (N6), metro status and the missing flag (N8). Built once per HCP, then repeated for every cohort when saved (N9).

In [0]:
# N7: deprivation index, mean and sd over one row per ZCTA
zcta_rows = acs.dropDuplicates(["ZCTA"])
st = zcta_rows.agg(*[F.mean(c).alias(f"{c}__mu") for c in DEP_COLS], *[F.stddev(c).alias(f"{c}__sd") for c in DEP_COLS]).collect()[0]
dep_expr = reduce(lambda a, b: a + b, [F.lit(DEP_SIGN.get(c, 1)) * (F.col(c) - F.lit(st[f"{c}__mu"])) / F.lit(st[f"{c}__sd"]) for c in DEP_COLS]) / F.lit(len(DEP_COLS))
acs_f = acs.withColumn("SDOH_DEPRIVATION_INDEX", dep_expr)
print("Deprivation index z-scored over", zcta_rows.count(), "ZCTAs:",
      {c: (round(st[f'{c}__mu'], 2), round(st[f'{c}__sd'], 2)) for c in DEP_COLS})

# N6: CR1 territory and county per ZIP (frozen Q2 2026 file), one row per ZIP
zt_raw = spark.read.csv(ZIP_TERR_PATH, header=True)
zip_terr = (zt_raw.filter(F.col("SEGMENT_ID") == "CR1")
            .select(F.lpad(F.col("ZIP"), 5, "0").alias("ZIPCODE"), F.col("GEO_NUM").alias("TERR_ID"), "COUNTY_NAME", "STATE")
            .dropDuplicates(["ZIPCODE"]))
area = acs_f.join(zip_terr, "ZIPCODE", "inner")
# Territory means over one row per ZCTA in the territory (unweighted)
terr = (area.dropDuplicates(["ZCTA", "TERR_ID"]).groupBy("TERR_ID").agg(
    F.mean("ACS_PCT_LT_HS_ZC").alias("TERR_SDOH_LT_HS"), F.mean("ACS_PCT_WHITE_ZC").alias("TERR_SDOH_WHITE"),
    F.mean("ACS_PCT_HISPANIC_ZC").alias("TERR_SDOH_HISPANIC"), F.mean("SDOH_DEPRIVATION_INDEX").alias("TERR_DEPRIVATION_MEAN")))
# County means over one row per ZCTA in the county; COUNTY_DEP_POPW weighted by population (N5, N12)
w_ok = F.col("SDOH_DEPRIVATION_INDEX").isNotNull() & (F.col(POP_COL) > 0)
county = (area.dropDuplicates(["ZCTA", "COUNTY_NAME", "STATE"]).groupBy("COUNTY_NAME", "STATE").agg(
    F.mean("ACS_PCT_LT_HS_ZC").alias("COUNTY_SDOH_LT_HS"), F.mean("ACS_PCT_WHITE_ZC").alias("COUNTY_SDOH_WHITE"),
    (F.sum(F.when(w_ok, F.col("SDOH_DEPRIVATION_INDEX") * F.col(POP_COL))) / F.sum(F.when(w_ok, F.col(POP_COL)))).alias("COUNTY_DEP_POPW")))
cbsa = (spark.table(CBSA_GEO).select(F.lpad(F.col("ZIP_CODE").cast("string"), 5, "0").alias("ZIPCODE"), "CBSA_CODE")
        .dropDuplicates(["ZIPCODE"]).withColumn("_in_geo", F.lit(1)))

feat_hcp = (spine.join(acs_f, "ZIPCODE", "left")
            .join(zip_terr, "ZIPCODE", "left").join(terr, "TERR_ID", "left").join(county, ["COUNTY_NAME", "STATE"], "left")
            .join(cbsa, "ZIPCODE", "left")
            # N8: missing states
            .withColumn("SDOH_MISSING_FLAG", F.when(F.col("ACS_PCT_LT_HS_ZC").isNull(), 1).otherwise(0))
            .withColumn("NO_CBSA_FLAG", F.when(F.col("ZIPCODE").isNull(), F.lit(None).cast("int"))
                        .when(F.col("_in_geo").isNull() | F.col("CBSA_CODE").isNull() | (F.trim(F.col("CBSA_CODE")) == ""), 1).otherwise(0))
            .withColumn("LOG_POP", F.log1p(F.col(POP_COL))))
feat_hcp = feat_hcp.select("PROVIDER_ID", "ZIPCODE", "TERR_ID", *FEATURE_COLS)
feat_pd = feat_hcp.toPandas()
print(f"{len(feat_pd):,} HCP rows, {feat_pd.PROVIDER_ID.nunique():,} distinct HCPs, {len(FEATURE_COLS)} features")

In [0]:
fp = feat_pd
record("S3 one row per HCP", len(fp) == n_hcp and fp.PROVIDER_ID.is_unique, f"{len(fp):,} rows, unique PROVIDER_ID: {fp.PROVIDER_ID.is_unique}")

# Coverage vs 08_2 section 2 (LT_HS value, territory, CBSA)
cov2 = {"lt_hs": int(fp.ACS_PCT_LT_HS_ZC.notna().sum()), "terr": int(fp.TERR_ID.notna().sum()),
        "in_cbsa": int((fp.NO_CBSA_FLAG == 0).sum()), "no_cbsa": int((fp.NO_CBSA_FLAG == 1).sum()),
        "missing_flag": int(fp.SDOH_MISSING_FLAG.sum())}
exp2 = {"lt_hs": 281978, "terr": 282872, "in_cbsa": 269732, "no_cbsa": 13149, "missing_flag": n_hcp - 281978}
record("S3 coverage = 08_2", cov2 == exp2, f"{cov2} (expected {exp2})")

# N8: no-ZIP HCPs have every ZIP / area feature null and SDOH_MISSING_FLAG = 1
no_zip = fp[fp.ZIPCODE.isna()]
non_flag = [c for c in FEATURE_COLS if c not in FLAG_COLS]
viol = int(no_zip[non_flag].notna().any(axis=1).sum()) + int((no_zip.SDOH_MISSING_FLAG != 1).sum()) + int(no_zip.NO_CBSA_FLAG.notna().sum())
flag_viol = int((~fp.SDOH_MISSING_FLAG.isin([0, 1])).sum()) + int((~fp.NO_CBSA_FLAG.dropna().isin([0, 1])).sum())
record("S3 missing-state rules (N8)", viol == 0 and flag_viol == 0,
       f"{len(no_zip):,} no-ZIP HCPs; violations: {viol}; flags outside 0/1: {flag_viol}")

# Percent columns within [0, 100]
pct_cols = [c for c in FEATURE_COLS if c.startswith("ACS_PCT_") or c in ("TERR_SDOH_LT_HS", "TERR_SDOH_WHITE", "TERR_SDOH_HISPANIC", "COUNTY_SDOH_LT_HS", "COUNTY_SDOH_WHITE")]
out_of_range = {c: int(((fp[c] < 0) | (fp[c] > 100)).sum()) for c in pct_cols}
record("S3 percent columns in [0,100]", sum(out_of_range.values()) == 0,
       f"{len(pct_cols)} columns, values outside [0,100]: {sum(out_of_range.values())}")

# N2: the only HCPs with a ZIP but no ACS row are those whose ZIP is not in acs20
zip_not_in_file = int((fp.ZIPCODE.notna() & ~fp.ZIPCODE.isin(set(acs.select('ZIPCODE').toPandas().ZIPCODE))).sum())
record("S3 ZIPs not in acs20", zip_not_in_file == 24, f"{zip_not_in_file} HCPs have a ZIP that is not in acs20 (08_2: 24)")

## Phase 2: Signal restated on KER-10 writers (N11)

Same window as the colocation notebook (its N13): cohort c17, look-back 2025-07-01 to 2025-09-30, outcome = counted by KER-10 `active_nbrx_writers` in any month 2025-10 to 2026-09. New-writer view = HCPs not counted by KER-10 in the look-back. Numeric features: writer rate in Q1 vs Q5 and Spearman rho; flags: rate with vs without. The 08_2 rho (LRx V7 outcome) is shown for comparison; the check is that the **direction** holds for every feature with |rho| >= 0.03 in 08_2.

In [0]:
SIG_COHORT = "c17"
row = cohorts[cohorts.cohort_id == SIG_COHORT].iloc[0]
lb_start = (pd.Timestamp(row.lookback_end_date) - pd.DateOffset(months=2)).date()
lb_end = (pd.Timestamp(row.lookback_end_date) + pd.offsets.MonthEnd(0)).date()
out_start = (pd.Timestamp(row.lookback_end_date) + pd.DateOffset(months=1)).date()
out_end = (pd.Timestamp(out_start) + pd.DateOffset(months=11)).date()

mv_rows = spark.sql(f"""
    SELECT CAST(`provider_id` AS STRING) AS PROVIDER_ID, `week_ending`, `month`, MEASURE(`active_nbrx_writers`) AS W
    FROM {MV_NBRX}
    WHERE `week_ending` >= DATE'{lb_start}'
    GROUP BY ALL
    HAVING W > 0""")
mv_max = mv_rows.agg(F.max("month")).collect()[0][0]
assert mv_max >= out_end, f"metric view ends {mv_max}, outcome needs {out_end}"
own_lb = (mv_rows.filter((F.col("week_ending") >= F.lit(lb_start)) & (F.col("week_ending") <= F.lit(lb_end)))
          .select("PROVIDER_ID").distinct().withColumn("OWN_WRITER_3M", F.lit(1)))
target = (mv_rows.filter((F.col("month") >= F.lit(out_start)) & (F.col("month") <= F.lit(out_end)))
          .select("PROVIDER_ID").distinct().withColumn("TARGET", F.lit(1)))
sig_base = universe.join(own_lb, "PROVIDER_ID", "left").join(target, "PROVIDER_ID", "left").fillna(0, ["OWN_WRITER_3M", "TARGET"]).toPandas()
sd = fp.merge(sig_base, on="PROVIDER_ID", how="left")
base_rate, nw = sd.TARGET.mean(), sd[sd.OWN_WRITER_3M == 0]
record("P1 KER-10 outcome built", len(sd) == n_hcp and nw.TARGET.mean() < base_rate,
       f"{SIG_COHORT}: look-back {lb_start}..{lb_end}, outcome {out_start}..{out_end}; writers {int(sd.TARGET.sum()):,} ({base_rate:.2%}); "
       f"new-writer view {len(nw):,} HCPs ({nw.TARGET.mean():.2%})")

In [0]:
# rho reported in 08_2 (LRx V7 outcome, sections 3.1-3.4); flags: sign of (rate yes - rate no)
RHO_08_2 = {"ACS_PCT_ASIAN_ZC": 0.021, "ACS_PCT_BLACK_ZC": 0.040, "ACS_PCT_HISPANIC_ZC": 0.058, "ACS_PCT_OTHER_RACE_ZC": 0.056,
            "ACS_PCT_EDUC_ZC": -0.011, "ACS_PCT_TRANSPORT_ZC": 0.031, "ACS_PCT_UNEMPLOY_ZC": 0.044, "ACS_PER_CAPITA_INC_ZC": -0.039,
            "ACS_PCT_POV_ASIAN_ZC": 0.022, "ACS_PCT_POV_BLACK_ZC": 0.016, "ACS_PCT_POV_HISPANIC_ZC": 0.024, "ACS_PCT_POV_WHITE_ZC": 0.036,
            "ACS_PCT_COLLEGE_ASSOCIATE_DGR_ZC": -0.014, "ACS_PCT_BACHELOR_DGR_ZC": -0.025, "ACS_PCT_GRADUATE_DGR_ZC": -0.026,
            "ACS_PCT_HS_GRADUATE_ZC": 0.002, "ACS_PCT_LT_HS_ZC": 0.062, "ACS_PCT_PUBL_TRANSIT_ZC": 0.015, "ACS_PCT_UNINSURED_ZC": 0.050,
            "ACS_PCT_UNINSURED_BELOW64_ZC": 0.049, "ACS_PCT_WHITE_NONHISP_ZC": -0.085,
            "TERR_SDOH_LT_HS": 0.084, "TERR_SDOH_WHITE": -0.079, "COUNTY_DEP_POPW": 0.076,
            "ACS_PCT_MEDICAID_ANY_ZC": 0.046, "ACS_PCT_MARRIED_SP_AB_F_ZC": 0.074, "ACS_PCT_1UP_PERS_1ROOM_ZC": 0.065,
            "ACS_PCT_SELF_MDCR_ABOVE35_ZC": -0.060, "SDOH_DEPRIVATION_INDEX": 0.058, "TERR_DEPRIVATION_MEAN": 0.059,
            "TERR_SDOH_HISPANIC": 0.060, "COUNTY_SDOH_LT_HS": 0.083, "COUNTY_SDOH_WHITE": -0.075, "LOG_POP": 0.044,
            "SDOH_MISSING_FLAG": -1, "NO_CBSA_FLAG": -1}   # flags: 08_2 rate yes < rate no (4.8-5.0% vs ~9.9%; 7.3% vs 10.0%)

def spearman(x, y):
    return float(pd.Series(x).rank().corr(pd.Series(y).rank()))

rows = []
for c in FEATURE_COLS:
    s = sd[[c, "TARGET", "OWN_WRITER_3M"]].dropna(subset=[c])
    s_nw = s[s.OWN_WRITER_3M == 0]
    if c in FLAG_COLS:
        a, b = s[s[c] == 1].TARGET.mean(), s[s[c] == 0].TARGET.mean()
        na, nb = s_nw[s_nw[c] == 1].TARGET.mean(), s_nw[s_nw[c] == 0].TARGET.mean()
        kind, rho = "yes vs no", spearman(s[c], s.TARGET)
    else:
        q = pd.qcut(s[c].rank(method="first"), 5, labels=False)
        qn = pd.qcut(s_nw[c].rank(method="first"), 5, labels=False)
        a, b = s[q == 0].TARGET.mean(), s[q == 4].TARGET.mean()
        na, nb = s_nw[qn == 0].TARGET.mean(), s_nw[qn == 4].TARGET.mean()
        kind, rho = "Q1 vs Q5", spearman(s[c], s.TARGET)
    ref = RHO_08_2[c]
    if c in FLAG_COLS:
        same_dir = np.sign(a - b) == np.sign(ref)
    else:
        same_dir = (abs(ref) < 0.03) or (np.sign(rho) == np.sign(ref))
    rows.append({"feature": c, "status": "Build" if c in BUILD_COLS else "Test", "fill_%": round(100 * len(s) / n_hcp, 1),
                 "compare": kind, "rate_A_%": round(100 * a, 1), "rate_B_%": round(100 * b, 1),
                 "new_writer_A_%": round(100 * na, 1), "new_writer_B_%": round(100 * nb, 1),
                 "rho_KER10": round(rho, 3), "rho_08_2": ref if c not in FLAG_COLS else None, "direction_holds": bool(same_dir)})
sig = pd.DataFrame(rows)
flips = sig[~sig.direction_holds].feature.tolist()
record("P2 signal direction holds on KER-10", not flips,
       f"{len(sig)} features; direction differs from 08_2 for: {flips if flips else 'none'} (features with |rho_08_2| < 0.03 not tested)")
display(sig)

## Phase 3: Save one row per HCP x cohort and check the schema on two runs

In [0]:
def expand(feat_df, cohorts_df):
    """N9: repeat the static HCP features for every cohort, with the colocation keys."""
    return (feat_df.crossJoin(F.broadcast(cohorts_df.select("cohort_id", F.col("lookback_end_date").alias("SVC_DT_M"))))
            .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "cohort_id"))
            .withColumn("PULL_DT", F.current_date()))

OUT_COLS = KEYS + FEATURE_COLS + ["PULL_DT"]
out = expand(feat_hcp.select("PROVIDER_ID", *FEATURE_COLS), coh).select(*OUT_COLS)
out.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(OUTPUT_PATH)
run1 = spark.read.format("delta").load(OUTPUT_PATH)
n_out = run1.count()
record("Output written", n_out == n_hcp * len(cohorts), f"{n_out:,} rows (= {n_hcp:,} x {len(cohorts)}) -> {OUTPUT_PATH}")
n_keys = run1.select("HCP_COHORT_ID").distinct().count()
record("S6 one row per HCP x cohort", n_keys == n_out, f"distinct HCP_COHORT_ID {n_keys:,}")

# Second run: rebuild the last 2 cohorts from the same code path and compare schema and values
last2 = cohorts.cohort_id.iloc[-2:].tolist()
run2 = expand(feat_hcp.select("PROVIDER_ID", *FEATURE_COLS), coh.filter(F.col("cohort_id").isin(last2))).select(*OUT_COLS)
same_schema = [(f.name, f.dataType.simpleString()) for f in run1.schema] == [(f.name, f.dataType.simpleString()) for f in run2.schema]
# Means and sds are recomputed on the rerun, so doubles can differ in the last bits: compare rounded to 9 decimals
def _r(df):
    return df.drop("PULL_DT").select(*[F.round(c, 9).alias(c) if t == "double" else F.col(c) for c, t in df.drop("PULL_DT").dtypes])
diff = _r(run1.filter(F.col("cohort_id").isin(last2))).exceptAll(_r(run2)).count()
record("S6 fixed schema on two runs", same_schema and diff == 0 and run1.columns == OUT_COLS,
       f"{len(OUT_COLS)} columns ({len(FEATURE_COLS)} features), schema identical: {same_schema}, differing rows on rerun of {last2}: {diff}")

# Static check: every HCP has identical feature values in all cohorts
hash_col = F.xxhash64(*[F.coalesce(F.col(c).cast("string"), F.lit("<null>")) for c in FEATURE_COLS])
n_var = run1.groupBy("PROVIDER_ID").agg(F.countDistinct(hash_col).alias("h")).filter("h > 1").count()
record("S6 values identical across cohorts", n_var == 0, f"HCPs whose values differ between cohorts: {n_var}")

In [0]:
def ok(*ids):
    return all(CHECKS[i]["passed"] for i in ids)
goals = [
    ("Source swap: acs20 replaces the CSV with identical values (N1)", ok("S1 acs20 one row per ZIPCODE", "N1 acs20 = Walgreens CSV")),
    ("ZIPCODE join fix recovers the 4,443 HCPs; coverage = 08_2 (N2, N3)", ok("S2 one rank-1 ZIP per BH_ID", "S2 coverage = 08_2 section 2", "S3 coverage = 08_2")),
    ("Missing states and value ranges (N8)", ok("S3 one row per HCP", "S3 missing-state rules (N8)", "S3 percent columns in [0,100]")),
    ("Signal direction from 08_2 holds on KER-10 writers (N11)", ok("P1 KER-10 outcome built", "P2 signal direction holds on KER-10")),
    ("One row per HCP x cohort, fixed schema, static values (N9)", ok("Output written", "S6 one row per HCP x cohort", "S6 fixed schema on two runs", "S6 values identical across cohorts")),
]
goal_df = pd.DataFrame(goals, columns=["goal", "achieved"])
check_df = pd.DataFrame([{"check": k, **v} for k, v in CHECKS.items()])
print(f"{int(check_df.passed.sum())} of {len(check_df)} checks passed")
display(check_df); display(goal_df)

In [0]:
final = spark.read.format("delta").load(OUTPUT_PATH)
last_c = cohorts.cohort_id.iloc[-1]
fr = final.filter(F.col("cohort_id") == last_c).select(
    *[F.avg(F.col(c).isNotNull().cast("double")).alias(f"{c}__filled") for c in FEATURE_COLS],
    *[F.avg((F.col(c) != 0).cast("double")).alias(f"{c}__nonzero") for c in FEATURE_COLS]).toPandas().T[0]
fill = pd.DataFrame({"feature": FEATURE_COLS, "status": ["Build" if c in BUILD_COLS else "Test" for c in FEATURE_COLS],
                     "filled_%": [round(100 * fr[f"{c}__filled"], 2) for c in FEATURE_COLS],
                     "non_zero_%": [round(100 * fr[f"{c}__nonzero"], 2) for c in FEATURE_COLS]})
print(f"{final.count():,} rows x {len(final.columns)} columns | fill rates for cohort {last_c} ({n_hcp:,} HCPs)")
display(fill)